# Logistic Regression from Scratch

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## Loading the Dataset

The Breast Cancer Wisconsin dataset is a binary classification dataset containing measurements computed from digitized images of breast tissue. Each observation is labeled as either malignant or benign. This dataset is commonly used to evaluate binary classification algorithms because it is well-structured and included with Scikit-learn.

In [ ]:
data = load_breast_cancer()

X = data.data
y = data.target

print(X.shape)
print(y.shape)

## Train-Test Split

To evaluate the model's ability to generalize to unseen data, the dataset is divided into training and testing sets. The model is trained only on the training data, while the test set is reserved for evaluation after training has been completed.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## Feature Scaling

Logistic regression converges more efficiently when the input features are on similar scales. Standardization transforms each feature to have a mean of zero and a standard deviation of one.

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Sigmoid Function

The sigmoid function converts any real-valued input into a probability between 0 and 1. Logistic regression uses this function to estimate the probability that an observation belongs to the positive class.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## Binary Cross-Entropy Loss

Binary cross-entropy measures how well the predicted probabilities match the true class labels. Lower values indicate better predictions, while larger values correspond to greater prediction error.

In [ ]:
def binary_cross_entropy(y_true, y_pred):

    epsilon = 1e-15

    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    loss = -np.mean(
        y_true * np.log(y_pred) +
        (1 - y_true) * np.log(1 - y_pred)
    )

    return loss

## Logistic Regression Class

The model learns a weight for each input feature and a bias term. During training, gradient descent updates these parameters to minimize the binary cross-entropy loss.

In [ ]:
class LogisticRegressionScratch:

    def __init__(self, learning_rate=0.01, epochs=1000):

        self.learning_rate = learning_rate
        self.epochs = epochs

    def fit(self, X, y):

        samples, features = X.shape

        self.weights = np.zeros(features)
        self.bias = 0

        self.losses = []

        for epoch in range(self.epochs):

            linear = np.dot(X, self.weights) + self.bias

            predictions = sigmoid(linear)

            loss = binary_cross_entropy(y, predictions)
            self.losses.append(loss)

            dw = (1 / samples) * np.dot(X.T, (predictions - y))
            db = (1 / samples) * np.sum(predictions - y)

            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

    def predict_proba(self, X):

        return sigmoid(np.dot(X, self.weights) + self.bias)

    def predict(self, X):

        probabilities = self.predict_proba(X)

        return (probabilities >= 0.5).astype(int)

## Training the Model

After implementing the learning algorithm, the model is trained using the training dataset. During training, the binary cross-entropy loss should gradually decrease as the model improves its predictions.

In [ ]:
model = LogisticRegressionScratch(
    learning_rate=0.01,
    epochs=1000
)

model.fit(X_train, y_train)

## Visualizing the Training Loss

Monitoring the loss over time helps determine whether gradient descent is successfully minimizing the objective function.

In [ ]:
plt.plot(model.losses)

plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Training Loss")
plt.show()

## Evaluating the Model

Once training is complete, the model is evaluated on the test dataset. Classification accuracy measures the proportion of correctly classified observations.

In [ ]:
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print(f"Scratch Accuracy: {accuracy:.4f}")

## Comparison with Scikit-learn

To verify the correctness of the implementation, the model is compared with Scikit-learn's optimized logistic regression implementation using the same training and testing data.

In [ ]:
sk_model = LogisticRegression(max_iter=1000)

sk_model.fit(X_train, y_train)

sk_predictions = sk_model.predict(X_test)

sk_accuracy = accuracy_score(y_test, sk_predictions)

print(f"Scikit-learn Accuracy: {sk_accuracy:.4f}")